# SparseGPT quick test for HGRN-340M

This notebook follows the same general structure as the attached `sparsegpt_opt.ipynb`, but it uses the repository-style `hgrn.py` wrapper and a substantially smaller HGRN checkpoint:

```text
m-a-p/340M-20B-HGRN-pure-baseline
```

The checkpoint uses the modern FLA `HGRNForCausalLM` architecture expected by `hgrn.py`. Its hidden size is 1,024, compared with 2,048 for the earlier 1.3B checkpoint, while both models have 24 recurrent blocks. The smaller hidden size substantially reduces the Hessian cost inside SparseGPT.

This notebook defaults to:

```text
Model: m-a-p/340M-20B-HGRN-pure-baseline
Calibration dataset: WikiText-2
Calibration samples: 8
Sparsity: 50%
```

These settings are intended to verify that loading, activation collection, pruning, evaluation, saving, and reloading all work. After the quick test succeeds, increase `NSAMPLES` to 32 or 128 for a stronger experiment.

The notebook will:

1. Check the GPU.
2. Install HGRN/FLA and Hugging Face dependencies.
3. Clone the original SparseGPT repository.
4. Replace only `datautils.py` with the updated dataset loader.
5. Copy the supplied `hgrn.py` into the repository without changing it.
6. Run HGRN-340M pruning through the same command-line interface as `opt.py`, `bloom.py`, and `llama.py`.
7. Save the tokenizer with the sparse checkpoint.
8. Verify the saved zero-weight sparsity.
9. Report dense and sparse WikiText-2 perplexity.
10. Run dense and sparse zero-shot LM Evaluation Harness tasks.
11. Compare dense and sparse generations.

> A CUDA GPU with roughly 12–16 GB or more of memory should be a much more comfortable starting point for this 340M test than the 1.3B checkpoint.


## Install dependencies

In [1]:
import subprocess
import sys

packages = [
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "datasets>=3.3,<5",
    "accelerate",
    "safetensors",
    "sentencepiece",
    "einops",
    "ninja",
    "flash-linear-attention",
    "lm_eval[hf]",
    "pandas",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        *packages,
    ],
    check=True,
)

print("Dependencies installed.")


Dependencies installed.


## Check the GPU

Do not continue unless `CUDA available` is `True`.

HGRN-340M is stored in BF16, so a GPU with BF16 support is strongly recommended.


In [2]:
import platform
import torch

print("Python platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is available.")

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GiB",
)

!nvidia-smi


Python platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: Tesla T4
BF16 supported: True
GPU memory: 14.56 GiB
Mon Jul 20 16:51:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             28W /   70W |     107MiB /  153

## Verify the installed libraries

In [3]:
import datasets
import fla
import tokenizers
import transformers
import torch

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Datasets:", datasets.__version__)
print("FLA:", getattr(fla, "__version__", "installed"))


Torch: 2.11.0+cu128
Transformers: 4.57.6
Tokenizers: 0.22.1
Datasets: 4.8.5
FLA: 0.5.1


## Clone the original SparseGPT repository

In [4]:
from pathlib import Path
import subprocess

# Works in both Google Colab and an ordinary Jupyter/Lambda environment.
WORKSPACE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKSPACE / "sparsegpt"

if REPO_DIR.exists():
    print("Repository already exists:", REPO_DIR)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/IST-DASLab/sparsegpt.git",
            str(REPO_DIR),
        ],
        check=True,
    )

print("SparseGPT repository:", REPO_DIR)


SparseGPT repository: /content/sparsegpt


## Enter the repository and create the output directory

In [5]:
import os

os.chdir(REPO_DIR)

OUTPUT_ROOT = REPO_DIR / "sparse_hgrn"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Current directory:", Path.cwd())
print("Output directory:", OUTPUT_ROOT)


Current directory: /content/sparsegpt
Output directory: /content/sparsegpt/sparse_hgrn


## Update only `datautils.py`

The original SparseGPT dataset loaders reference retired Hugging Face dataset scripts.

This cell replaces only `datautils.py` so that:

- WikiText-2 uses `Salesforce/wikitext`
- PTB uses its original raw text files
- C4 uses the same individual train and validation shards through the generic JSON loader

The HGRN wrapper and SparseGPT pruning implementation are not modified by this cell.


In [6]:
from pathlib import Path

datautils_source = 'import random\n\nimport numpy as np\nimport torch\nfrom datasets import load_dataset\nfrom transformers import AutoTokenizer, LlamaTokenizer\n\n\nWIKITEXT_REPO = "Salesforce/wikitext"\nWIKITEXT_CONFIG = "wikitext-2-raw-v1"\n\n# Same PTB source files used by the retired Hugging Face ptb_text_only loader.\nPTB_FILES = {\n    "train": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.train.txt",\n    "validation": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.valid.txt",\n    "test": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt",\n}\n\n# Same single C4 train and validation shards used by SparseGPT\'s old loader.\nC4_TRAIN_FILE = (\n    "https://huggingface.co/datasets/allenai/c4/resolve/main/"\n    "en/c4-train.00000-of-01024.json.gz"\n)\nC4_VALIDATION_FILE = (\n    "https://huggingface.co/datasets/allenai/c4/resolve/main/"\n    "en/c4-validation.00000-of-00008.json.gz"\n)\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.random.manual_seed(seed)\n\n\ndef get_tokenizer(model):\n    if "llama" in model.lower():\n        try:\n            tokenizer = LlamaTokenizer.from_pretrained(model, use_fast=False)\n        except Exception:\n            tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)\n\n        if tokenizer.bos_token_id != 1 or tokenizer.eos_token_id != 2:\n            try:\n                tokenizer.bos_token_id = 1\n                tokenizer.eos_token_id = 2\n            except AttributeError:\n                pass\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)\n\n    return tokenizer\n\n\ndef _sample_token_windows(token_ids, nsamples, seed, seqlen):\n    if token_ids.shape[1] <= seqlen:\n        raise ValueError(\n            f"Dataset contains only {token_ids.shape[1]} tokens, "\n            f"but seqlen={seqlen}."\n        )\n\n    rng = random.Random(seed)\n    trainloader = []\n\n    for _ in range(nsamples):\n        start = rng.randint(0, token_ids.shape[1] - seqlen - 1)\n        end = start + seqlen\n\n        inp = token_ids[:, start:end]\n        tar = inp.clone()\n        tar[:, :-1] = -100\n        trainloader.append((inp, tar))\n\n    return trainloader\n\n\ndef get_wikitext2(nsamples, seed, seqlen, model, tokenizer):\n    traindata = load_dataset(\n        WIKITEXT_REPO,\n        WIKITEXT_CONFIG,\n        split="train",\n    )\n    testdata = load_dataset(\n        WIKITEXT_REPO,\n        WIKITEXT_CONFIG,\n        split="test",\n    )\n\n    trainenc = tokenizer(" ".join(traindata["text"]), return_tensors="pt")\n    testenc = tokenizer("\\n\\n".join(testdata["text"]), return_tensors="pt")\n\n    trainloader = _sample_token_windows(\n        trainenc.input_ids,\n        nsamples=nsamples,\n        seed=seed,\n        seqlen=seqlen,\n    )\n\n    return trainloader, testenc\n\n\ndef get_ptb(nsamples, seed, seqlen, model, tokenizer):\n    # The generic text loader creates a column named "text".\n    dataset = load_dataset("text", data_files=PTB_FILES)\n\n    trainenc = tokenizer(\n        " ".join(dataset["train"]["text"]),\n        return_tensors="pt",\n    )\n    testenc = tokenizer(\n        " ".join(dataset["test"]["text"]),\n        return_tensors="pt",\n    )\n\n    trainloader = _sample_token_windows(\n        trainenc.input_ids,\n        nsamples=nsamples,\n        seed=seed,\n        seqlen=seqlen,\n    )\n\n    return trainloader, testenc\n\n\ndef get_c4(nsamples, seed, seqlen, model, tokenizer):\n    # Use the generic JSON loader rather than the retired C4 dataset script.\n    # Streaming prevents Colab from downloading the whole shard before sampling.\n    traindata = load_dataset(\n        "json",\n        data_files={"train": C4_TRAIN_FILE},\n        split="train",\n        streaming=True,\n    ).shuffle(seed=seed, buffer_size=10_000)\n\n    rng = random.Random(seed)\n    trainloader = []\n\n    for example in traindata:\n        text = example.get("text", "")\n        if not text:\n            continue\n\n        trainenc = tokenizer(text, return_tensors="pt")\n        token_count = trainenc.input_ids.shape[1]\n\n        if token_count <= seqlen:\n            continue\n\n        start = rng.randint(0, token_count - seqlen - 1)\n        end = start + seqlen\n\n        inp = trainenc.input_ids[:, start:end]\n        tar = inp.clone()\n        tar[:, :-1] = -100\n        trainloader.append((inp, tar))\n\n        if len(trainloader) == nsamples:\n            break\n\n    if len(trainloader) < nsamples:\n        raise RuntimeError(\n            f"Only found {len(trainloader)} usable C4 samples; "\n            f"requested {nsamples}."\n        )\n\n    # Preserve SparseGPT\'s original C4 evaluation length: 256 * seqlen tokens.\n    validation_target_tokens = 256 * seqlen\n    valdata = load_dataset(\n        "json",\n        data_files={"validation": C4_VALIDATION_FILE},\n        split="validation",\n        streaming=True,\n    )\n\n    chunks = []\n    token_count = 0\n\n    for example in valdata:\n        text = example.get("text", "")\n        if not text:\n            continue\n\n        ids = tokenizer(text, return_tensors="pt").input_ids\n        if ids.numel() == 0:\n            continue\n\n        chunks.append(ids)\n        token_count += ids.shape[1]\n\n        if token_count >= validation_target_tokens:\n            break\n\n    if token_count < validation_target_tokens:\n        raise RuntimeError(\n            f"C4 validation produced {token_count} tokens; "\n            f"needed {validation_target_tokens}."\n        )\n\n    valenc_ids = torch.cat(chunks, dim=1)\n    valenc_ids = valenc_ids[:, :validation_target_tokens]\n\n    class TokenizerWrapper:\n        def __init__(self, input_ids):\n            self.input_ids = input_ids\n\n    return trainloader, TokenizerWrapper(valenc_ids)\n\n\ndef get_loaders(name, nsamples=128, seed=0, seqlen=2048, model=""):\n    tokenizer = get_tokenizer(model)\n\n    if "wikitext2" in name:\n        return get_wikitext2(nsamples, seed, seqlen, model, tokenizer)\n    if "ptb" in name:\n        return get_ptb(nsamples, seed, seqlen, model, tokenizer)\n    if "c4" in name:\n        return get_c4(nsamples, seed, seqlen, model, tokenizer)\n\n    raise ValueError(\n        f"Unknown dataset {name!r}. Choose \'wikitext2\', \'ptb\', or \'c4\'."\n    )\n'
datautils_file = REPO_DIR / "datautils.py"
datautils_file.write_text(datautils_source, encoding="utf-8")

print("Updated:", datautils_file)


Updated: /content/sparsegpt/datautils.py


## Copy the supplied `hgrn.py` into SparseGPT

The embedded source below is the exact supplied wrapper. The notebook checks its SHA-256 hash after writing it.


In [7]:
import hashlib
from pathlib import Path

hgrn_source = "import time\n\nimport torch\nimport torch.nn as nn\n\nfrom sparsegpt import *\nfrom modelutils import *\nfrom quant import *\n\ntry:\n    import wandb\n    has_wandb = True\nexcept:\n    has_wandb = False\n\n\ndef get_hgrn(model):\n    import torch\n\n    def skip(*args, **kwargs):\n        pass\n\n    torch.nn.init.kaiming_uniform_ = skip\n    torch.nn.init.uniform_ = skip\n    torch.nn.init.normal_ = skip\n\n    import fla\n    from fla.models.hgrn import HGRNForCausalLM\n\n    model = HGRNForCausalLM.from_pretrained(model, torch_dtype='auto')\n    model.seqlen = model.config.max_position_embeddings\n\n    if getattr(model.model, 'use_attnres', False):\n        raise NotImplementedError(\n            'This wrapper currently supports standard HGRN blocks without '\n            'attention-residual aggregation.'\n        )\n\n    return model\n\n\ndef get_hgrn_lower_bounds(model):\n    if not getattr(model.config, 'use_lower_bound', False):\n        return None\n\n    lower_bounds = model.model.lower_bounds.softmax(0, dtype=torch.float)\n    lower_bounds = lower_bounds.cumsum(0) - lower_bounds[0]\n    return lower_bounds\n\n\ndef hgrn_layer_forward(layer, inp, attention_mask, lower_bound):\n    return layer(\n        inp,\n        attention_mask=attention_mask,\n        past_key_values=None,\n        use_cache=False,\n        output_attentions=False,\n        lower_bound=lower_bound,\n        attnres_states=None,\n    )[0]\n\n\ndef add_batch(gpts, name):\n    def tmp(_, inp, out):\n        gpts[name].add_batch(inp[0].data, out.data)\n\n    return tmp\n\n\ndef add_fused_down_batch(gpts, name, layer):\n    from fla.modules.activations import powglu, swiglu\n\n    def tmp(_, inp, out):\n        gate = inp[0]\n        up = inp[1]\n\n        if layer.mlp.hidden_act == 'swish':\n            down_inp = swiglu(gate, up)\n        elif layer.mlp.hidden_act == 'powlu':\n            down_inp = powglu(\n                gate,\n                up,\n                layer.mlp.powglu_power,\n            )\n        else:\n            raise ValueError(\n                f'Unsupported HGRN MLP activation: {layer.mlp.hidden_act}'\n            )\n\n        gpts[name].add_batch(down_inp.data, out.data)\n\n    return tmp\n\n\ndef register_hgrn_hooks(layer, subset, gpts):\n    handles = []\n\n    for name in gpts:\n        if (\n            name == 'mlp.down_proj'\n            and getattr(layer.mlp, 'fuse_swiglu', False)\n        ):\n            if not hasattr(layer.mlp, 'swiglu_linear'):\n                raise RuntimeError(\n                    'The fused HGRN MLP does not expose swiglu_linear, so '\n                    'SparseGPT cannot collect down_proj inputs without '\n                    'changing the model forward path.'\n                )\n            handles.append(\n                layer.mlp.swiglu_linear.register_forward_hook(\n                    add_fused_down_batch(gpts, name, layer)\n                )\n            )\n        else:\n            handles.append(\n                subset[name].register_forward_hook(\n                    add_batch(gpts, name)\n                )\n            )\n\n    return handles\n\n\n@torch.no_grad()\ndef hgrn_sequential(model, dataloader, dev):\n    print('Starting ...')\n\n    use_cache = model.config.use_cache\n    model.config.use_cache = False\n    layers = model.model.layers\n\n    model.model.embeddings = model.model.embeddings.to(dev)\n    if getattr(model.config, 'use_lower_bound', False):\n        model.model.lower_bounds.data = model.model.lower_bounds.data.to(dev)\n    layers[0] = layers[0].to(dev)\n\n    dtype = next(iter(model.parameters())).dtype\n    inps = torch.zeros(\n        (args.nsamples, model.seqlen, model.config.hidden_size),\n        dtype=dtype,\n        device=dev,\n    )\n    cache = {'i': 0, 'attention_mask': None}\n\n    class Catcher(nn.Module):\n        def __init__(self, module):\n            super().__init__()\n            self.module = module\n\n        def forward(self, inp, **kwargs):\n            inps[cache['i']] = inp\n            cache['i'] += 1\n            cache['attention_mask'] = kwargs.get('attention_mask')\n            raise ValueError\n\n    layers[0] = Catcher(layers[0])\n    for batch in dataloader:\n        try:\n            model(batch[0].to(dev))\n        except ValueError:\n            pass\n    layers[0] = layers[0].module\n\n    layers[0] = layers[0].cpu()\n    model.model.embeddings = model.model.embeddings.cpu()\n    if getattr(model.config, 'use_lower_bound', False):\n        model.model.lower_bounds.data = model.model.lower_bounds.data.cpu()\n    torch.cuda.empty_cache()\n\n    outs = torch.zeros_like(inps)\n    attention_mask = cache['attention_mask']\n    lower_bounds = get_hgrn_lower_bounds(model)\n\n    print('Ready.')\n\n    quantizers = {}\n    for i in range(len(layers)):\n        layer = layers[i].to(dev)\n        full = find_layers(layer)\n\n        if args.true_sequential:\n            sequential = [\n                ['attn.i_proj', 'attn.f_proj', 'attn.g_proj'],\n                ['attn.o_proj'],\n                ['mlp.up_proj', 'mlp.gate_proj'],\n                ['mlp.down_proj'],\n            ]\n            sequential = [\n                [name for name in names if name in full]\n                for names in sequential\n            ]\n            sequential = [names for names in sequential if names]\n        else:\n            sequential = [list(full.keys())]\n\n        lower_bound = (\n            lower_bounds[i].to(dev)\n            if lower_bounds is not None\n            else None\n        )\n\n        for names in sequential:\n            subset = {name: full[name] for name in names}\n\n            gpts = {}\n            for name in subset:\n                selected = (\n                    args.minlayer <= i < args.maxlayer\n                    and args.prune_only in name\n                )\n                if args.invert:\n                    selected = not selected\n                if not selected:\n                    continue\n\n                gpts[name] = SparseGPT(subset[name])\n                if args.wbits < 16:\n                    gpts[name].quantizer = Quantizer()\n                    gpts[name].quantizer.configure(\n                        args.wbits,\n                        perchannel=True,\n                        sym=False,\n                        mse=False,\n                    )\n\n            handles = register_hgrn_hooks(\n                layer,\n                subset,\n                gpts,\n            )\n\n            for j in range(args.nsamples):\n                outs[j] = hgrn_layer_forward(\n                    layer,\n                    inps[j].unsqueeze(0),\n                    attention_mask,\n                    lower_bound,\n                )\n\n            for handle in handles:\n                handle.remove()\n\n            for name in gpts:\n                print(i, name)\n                print('Pruning ...')\n                gpts[name].fasterprune(\n                    args.sparsity,\n                    prunen=args.prunen,\n                    prunem=args.prunem,\n                    percdamp=args.percdamp,\n                    blocksize=args.blocksize,\n                )\n                gpts[name].free()\n\n        for j in range(args.nsamples):\n            outs[j] = hgrn_layer_forward(\n                layer,\n                inps[j].unsqueeze(0),\n                attention_mask,\n                lower_bound,\n            )\n\n        layers[i] = layer.cpu()\n        del layer\n        if 'gpts' in locals():\n            del gpts\n        torch.cuda.empty_cache()\n\n        inps, outs = outs, inps\n\n    model.config.use_cache = use_cache\n\n    return quantizers\n\n\n@torch.no_grad()\ndef hgrn_eval(model, testenc, dev, dataset: str, log_wandb: bool = False):\n    print('Evaluating ...')\n\n    testenc = testenc.input_ids\n    nsamples = testenc.numel() // model.seqlen\n\n    use_cache = model.config.use_cache\n    model.config.use_cache = False\n    layers = model.model.layers\n\n    model.model.embeddings = model.model.embeddings.to(dev)\n    if getattr(model.config, 'use_lower_bound', False):\n        model.model.lower_bounds.data = model.model.lower_bounds.data.to(dev)\n    layers[0] = layers[0].to(dev)\n\n    dtype = next(iter(model.parameters())).dtype\n    inps = torch.zeros(\n        (nsamples, model.seqlen, model.config.hidden_size),\n        dtype=dtype,\n        device=dev,\n    )\n    cache = {'i': 0, 'attention_mask': None}\n\n    class Catcher(nn.Module):\n        def __init__(self, module):\n            super().__init__()\n            self.module = module\n\n        def forward(self, inp, **kwargs):\n            inps[cache['i']] = inp\n            cache['i'] += 1\n            cache['attention_mask'] = kwargs.get('attention_mask')\n            raise ValueError\n\n    layers[0] = Catcher(layers[0])\n    for i in range(nsamples):\n        batch = testenc[\n            :,\n            (i * model.seqlen):((i + 1) * model.seqlen),\n        ].to(dev)\n        try:\n            model(batch)\n        except ValueError:\n            pass\n    layers[0] = layers[0].module\n\n    layers[0] = layers[0].cpu()\n    model.model.embeddings = model.model.embeddings.cpu()\n    if getattr(model.config, 'use_lower_bound', False):\n        model.model.lower_bounds.data = model.model.lower_bounds.data.cpu()\n    torch.cuda.empty_cache()\n\n    outs = torch.zeros_like(inps)\n    attention_mask = cache['attention_mask']\n    lower_bounds = get_hgrn_lower_bounds(model)\n\n    for i in range(len(layers)):\n        print(i)\n        layer = layers[i].to(dev)\n\n        if args.gmp:\n            subset = find_layers(layer)\n            for name in subset:\n                W = subset[name].weight.data\n                thresh = torch.sort(torch.abs(W.flatten()))[0][\n                    int(W.numel() * args.sparsity)\n                ]\n                W.data[torch.abs(W.data) <= thresh] = 0\n\n        lower_bound = (\n            lower_bounds[i].to(dev)\n            if lower_bounds is not None\n            else None\n        )\n\n        for j in range(nsamples):\n            outs[j] = hgrn_layer_forward(\n                layer,\n                inps[j].unsqueeze(0),\n                attention_mask,\n                lower_bound,\n            )\n\n        layers[i] = layer.cpu()\n        del layer\n        torch.cuda.empty_cache()\n        inps, outs = outs, inps\n\n    model.model.norm = model.model.norm.to(dev)\n    model.lm_head = model.lm_head.to(dev)\n\n    testenc = testenc.to(dev)\n    nlls = []\n    for i in range(nsamples):\n        hidden_states = inps[i].unsqueeze(0)\n        hidden_states = model.model.norm(hidden_states)\n        lm_logits = model.lm_head(hidden_states)\n        shift_logits = lm_logits[:, :-1, :].contiguous()\n        shift_labels = testenc[\n            :,\n            (i * model.seqlen):((i + 1) * model.seqlen),\n        ][:, 1:]\n        loss_fct = nn.CrossEntropyLoss()\n        loss = loss_fct(\n            shift_logits.view(-1, shift_logits.size(-1)),\n            shift_labels.view(-1),\n        )\n        neg_log_likelihood = loss.float() * model.seqlen\n        nlls.append(neg_log_likelihood)\n\n    ppl = torch.exp(\n        torch.stack(nlls).sum() / (nsamples * model.seqlen)\n    )\n    print(f'Perplexity: {ppl.item():3f}')\n    if log_wandb:\n        wandb.log({f'{dataset}/perplexity': ppl.item()})\n\n    model.config.use_cache = use_cache\n\n\nif __name__ == '__main__':\n    import argparse\n    from datautils import *\n\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\n        'model',\n        type=str,\n        help='HGRN model to load; pass a Hugging Face HGRN checkpoint.',\n    )\n    parser.add_argument(\n        'dataset',\n        type=str,\n        choices=['wikitext2', 'ptb', 'c4'],\n        help='Where to extract calibration data from.',\n    )\n    parser.add_argument(\n        '--seed',\n        type=int,\n        default=0,\n        help='Seed for sampling the calibration data.',\n    )\n    parser.add_argument(\n        '--nsamples',\n        type=int,\n        default=128,\n        help='Number of calibration data samples.',\n    )\n    parser.add_argument(\n        '--percdamp',\n        type=float,\n        default=.01,\n        help='Percent of the average Hessian diagonal to use for dampening.',\n    )\n    parser.add_argument(\n        '--sparsity',\n        type=float,\n        default=0,\n        help='Target sparsity',\n    )\n    parser.add_argument(\n        '--prunen',\n        type=int,\n        default=0,\n        help='N for N:M pruning.',\n    )\n    parser.add_argument(\n        '--prunem',\n        type=int,\n        default=0,\n        help='M for N:M pruning.',\n    )\n    parser.add_argument(\n        '--blocksize',\n        type=int,\n        default=128,\n        help='Blocksize to use for adaptive mask selection.',\n    )\n    parser.add_argument(\n        '--gmp',\n        action='store_true',\n        help='Whether to run the GMP baseline.',\n    )\n    parser.add_argument(\n        '--wbits',\n        type=int,\n        default=16,\n        help='Whether to quantize as well.',\n    )\n    parser.add_argument(\n        '--minlayer',\n        type=int,\n        default=-1,\n        help='Prune all layers with id >= this.',\n    )\n    parser.add_argument(\n        '--maxlayer',\n        type=int,\n        default=1000,\n        help='Prune all layers with id < this.',\n    )\n    parser.add_argument(\n        '--prune_only',\n        type=str,\n        default='',\n        help='Prune only layers that contain this text.',\n    )\n    parser.add_argument(\n        '--invert',\n        action='store_true',\n        help='Invert subset.',\n    )\n    parser.add_argument(\n        '--save',\n        type=str,\n        default='',\n        help='Path to saved model.',\n    )\n    parser.add_argument(\n        '--true-sequential',\n        action='store_true',\n        help='Whether to run in true sequential model.',\n    )\n    parser.add_argument(\n        '--log_wandb',\n        action='store_true',\n        help='Whether to log to wandb.',\n    )\n\n    args = parser.parse_args()\n\n    if args.log_wandb:\n        assert has_wandb, 'wandb not installed try `pip install wandb`'\n        wandb.init(config=args)\n\n    model = get_hgrn(args.model)\n    model.eval()\n\n    dataloader, testloader = get_loaders(\n        args.dataset,\n        nsamples=args.nsamples,\n        seed=args.seed,\n        model=args.model,\n        seqlen=model.seqlen,\n    )\n\n    if (args.sparsity or args.prunen) and not args.gmp:\n        tick = time.time()\n        hgrn_sequential(model, dataloader, DEV)\n        for name, parameter in model.named_parameters():\n            print(name, torch.mean((parameter == 0).float()))\n            if 'mlp.down_proj' in name:\n                break\n        print(time.time() - tick)\n\n    for dataset in ['wikitext2', 'ptb', 'c4']:\n        dataloader, testloader = get_loaders(\n            dataset,\n            seed=args.seed,\n            model=args.model,\n            seqlen=model.seqlen,\n        )\n        print('Dataset:', dataset)\n        hgrn_eval(\n            model,\n            testloader,\n            DEV,\n            dataset,\n            args.log_wandb,\n        )\n\n    if args.save:\n        model.save_pretrained(args.save)\n"
expected_sha256 = '6a08addb39812f521e585fdcc6a6d2368f1de5d2ae67e336919c73101e7d8b1e'

hgrn_file = REPO_DIR / "hgrn.py"
hgrn_file.write_text(hgrn_source, encoding="utf-8")

actual_sha256 = hashlib.sha256(
    hgrn_file.read_bytes()
).hexdigest()

print("Created:", hgrn_file)
print("Expected SHA-256:", expected_sha256)
print("Actual SHA-256:  ", actual_sha256)

assert actual_sha256 == expected_sha256
print("The notebook copied hgrn.py without changing it.")


Created: /content/sparsegpt/hgrn.py
Expected SHA-256: 6a08addb39812f521e585fdcc6a6d2368f1de5d2ae67e336919c73101e7d8b1e
Actual SHA-256:   6a08addb39812f521e585fdcc6a6d2368f1de5d2ae67e336919c73101e7d8b1e
The notebook copied hgrn.py without changing it.


## Compile-check the Python files

In [8]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(REPO_DIR / "hgrn.py"),
        str(REPO_DIR / "datautils.py"),
        str(REPO_DIR / "sparsegpt.py"),
        str(REPO_DIR / "modelutils.py"),
        str(REPO_DIR / "quant.py"),
    ],
    check=True,
)

print("All Python files passed the syntax check.")


All Python files passed the syntax check.


## Confirm that `hgrn.py` has the repository-style flags

In [9]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(REPO_DIR / "hgrn.py"), "--help"],
    cwd=REPO_DIR,
    check=True,
    text=True,
    capture_output=True,
)

print(result.stdout)


usage: hgrn.py [-h] [--seed SEED] [--nsamples NSAMPLES] [--percdamp PERCDAMP]
               [--sparsity SPARSITY] [--prunen PRUNEN] [--prunem PRUNEM]
               [--blocksize BLOCKSIZE] [--gmp] [--wbits WBITS]
               [--minlayer MINLAYER] [--maxlayer MAXLAYER]
               [--prune_only PRUNE_ONLY] [--invert] [--save SAVE]
               [--true-sequential] [--log_wandb]
               model {wikitext2,ptb,c4}

positional arguments:
  model                 HGRN model to load; pass a Hugging Face HGRN
                        checkpoint.
  {wikitext2,ptb,c4}    Where to extract calibration data from.

options:
  -h, --help            show this help message and exit
  --seed SEED           Seed for sampling the calibration data.
  --nsamples NSAMPLES   Number of calibration data samples.
  --percdamp PERCDAMP   Percent of the average Hessian diagonal to use for
                        dampening.
  --sparsity SPARSITY   Target sparsity
  --prunen PRUNEN       N for N:M prunin

## Test the updated WikiText-2 loader

This downloads only a tiny calibration sample and confirms that `datautils.py` works with the HGRN tokenizer.


In [10]:
import importlib
import sys

# Reload datautils in case an older version was imported earlier.
sys.path.insert(0, str(REPO_DIR))
import datautils
importlib.reload(datautils)

sample_loader, sample_test = datautils.get_loaders(
    "wikitext2",
    nsamples=2,
    seed=0,
    model="m-a-p/340M-20B-HGRN-pure-baseline",
    seqlen=128,
)

print("Calibration samples:", len(sample_loader))
print("Calibration shape:", sample_loader[0][0].shape)
print("Test token shape:", sample_test.input_ids.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Calibration samples: 2
Calibration shape: torch.Size([1, 128])
Test token shape: torch.Size([1, 334661])


# Pruning example

The command format is the same as the original SparseGPT scripts:

```bash
python hgrn.py MODEL DATASET [FLAGS]
```

Available calibration datasets:

- `wikitext2`
- `ptb`
- `c4`

Important flags:

- `--sparsity 0.5` applies 50% unstructured SparseGPT pruning.
- `--prunen 2 --prunem 4` applies 2:4 pruning.
- `--gmp` uses global magnitude pruning instead of SparseGPT.
- `--wbits 4` adds 4-bit weight quantization.
- `--nsamples 8` uses eight calibration sequences.
- `--true-sequential` prunes HGRN projections in forward-order groups.
- `--minlayer`, `--maxlayer`, `--prune_only`, and `--invert` select subsets.
- `--save PATH` saves the resulting checkpoint.

The wrapper evaluates WikiText-2, PTB, and C4 after pruning, just like the original repository scripts.


## Choose the model and pruning settings

In [11]:
MODEL_ID = "m-a-p/340M-20B-HGRN-pure-baseline"
CALIBRATION_DATASET = "wikitext2"

SPARSITY = 0.50
NSAMPLES = 8
SEED = 0
PERCDAMP = 0.01
BLOCKSIZE = 128

SPARSE_MODEL_DIR = OUTPUT_ROOT / "hgrn-340m-sparsegpt-50"

# Add optional repository-style flags here.
# Examples:
# EXTRA_FLAGS = ["--true-sequential"]
# EXTRA_FLAGS = ["--prunen", "2", "--prunem", "4"]
# EXTRA_FLAGS = ["--wbits", "4"]
EXTRA_FLAGS = []

print("Model:", MODEL_ID)
print("Calibration dataset:", CALIBRATION_DATASET)
print("Target sparsity:", SPARSITY)
print("Calibration samples:", NSAMPLES)
print("Output:", SPARSE_MODEL_DIR)


Model: m-a-p/340M-20B-HGRN-pure-baseline
Calibration dataset: wikitext2
Target sparsity: 0.5
Calibration samples: 8
Output: /content/sparsegpt/sparse_hgrn/hgrn-340m-sparsegpt-50


# Evaluation settings

There are two different evaluation stages:

1. **Perplexity:** WikiText-2 language-model perplexity for the dense and sparse checkpoints. Lower is better.
2. **LM Evaluation Harness:** zero-shot task accuracy for LAMBADA, PIQA, ARC-Easy, and ARC-Challenge. Higher is better.

The settings below are intentionally limited for a quick pipeline test.

- `PPL_MAX_SEGMENTS = 16` evaluates only the first 16 non-overlapping WikiText-2 segments.
- `LM_EVAL_LIMIT = 100` evaluates at most 100 examples from each zero-shot task.

For final results, set both limits to `None`.


In [12]:
# WikiText-2 perplexity settings
PPL_SEQUENCE_LENGTH = 512
PPL_MAX_SEGMENTS = 16       # Set to None for the full WikiText-2 test set.

# LM Evaluation Harness settings
LM_EVAL_TASKS = [
    "lambada_openai",
    "piqa",
    "arc_easy",
    "arc_challenge",
]
LM_EVAL_BATCH_SIZE = 4
LM_EVAL_MAX_LENGTH = 512
LM_EVAL_LIMIT = 100         # Set to None for full task evaluation.
LM_EVAL_NUM_FEWSHOT = 0

EVALUATION_DIR = OUTPUT_ROOT / "evaluation_results"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

print("Perplexity sequence length:", PPL_SEQUENCE_LENGTH)
print("Perplexity segment limit:", PPL_MAX_SEGMENTS)
print("LM-eval tasks:", LM_EVAL_TASKS)
print("LM-eval example limit per task:", LM_EVAL_LIMIT)
print("Evaluation output directory:", EVALUATION_DIR)


Perplexity sequence length: 512
Perplexity segment limit: 16
LM-eval tasks: ['lambada_openai', 'piqa', 'arc_easy', 'arc_challenge']
LM-eval example limit per task: 100
Evaluation output directory: /content/sparsegpt/sparse_hgrn/evaluation_results


### Why this should be faster

The 340M checkpoint uses hidden size 1,024 and 24 HGRN blocks. The prior 1.3B checkpoint uses hidden size 2,048 and 24 blocks. SparseGPT builds square Hessian matrices for linear-layer inputs, so halving the hidden width reduces the pruning cost by much more than the checkpoint-size difference alone.


# Explicit evaluation utilities

The original `hgrn.py` wrapper evaluates WikiText-2, PTB, and C4 automatically at the end of its command. Those perplexities appear in the pruning-cell console output.

The functions below add a separate, clearly labeled evaluation path that:

- evaluates only WikiText-2 when requested;
- uses identical code for the dense and sparse models;
- saves the numerical result to JSON;
- loads only one checkpoint on the GPU at a time.

For LM Evaluation Harness, the HGRN checkpoint is loaded through FLA first and then passed to the harness as a pre-initialized Hugging Face model.


In [13]:
import gc
import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
)

import fla
from fla.models.hgrn import HGRNConfig, HGRNForCausalLM


def register_hgrn_for_transformers():
    AutoConfig.register(
        "hgrn",
        HGRNConfig,
        exist_ok=True,
    )
    AutoModelForCausalLM.register(
        HGRNConfig,
        HGRNForCausalLM,
        exist_ok=True,
    )


register_hgrn_for_transformers()


def release_model(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()


def load_hgrn_checkpoint(model_path):
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        use_fast=False,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    ).to("cuda")
    model.eval()
    model.config.use_cache = False

    return model, tokenizer


@torch.inference_mode()
def evaluate_wikitext2_perplexity(
    model_path,
    sequence_length=512,
    max_segments=16,
):
    model, tokenizer = load_hgrn_checkpoint(model_path)

    test_data = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )
    text = "\n\n".join(test_data["text"])
    encoded = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids[0]

    available_segments = encoded.numel() // sequence_length
    segment_count = available_segments
    if max_segments is not None:
        segment_count = min(segment_count, max_segments)

    if segment_count < 1:
        release_model(model)
        raise RuntimeError(
            "WikiText-2 contains fewer tokens than the selected "
            "evaluation sequence length."
        )

    total_negative_log_likelihood = 0.0
    total_predicted_tokens = 0

    for segment_index in range(segment_count):
        start = segment_index * sequence_length
        stop = start + sequence_length
        input_ids = encoded[start:stop].unsqueeze(0).to("cuda")

        outputs = model(
            input_ids=input_ids,
            use_cache=False,
            return_dict=True,
        )
        logits = outputs.logits

        shift_logits = logits[:, :-1, :].float().contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        negative_log_likelihood = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="sum",
        )

        total_negative_log_likelihood += float(
            negative_log_likelihood.item()
        )
        total_predicted_tokens += int(shift_labels.numel())

        if (
            (segment_index + 1) % 4 == 0
            or segment_index + 1 == segment_count
        ):
            print(
                f"  WikiText-2 segments: "
                f"{segment_index + 1}/{segment_count}"
            )

        del (
            input_ids,
            outputs,
            logits,
            shift_logits,
            shift_labels,
            negative_log_likelihood,
        )

    perplexity = math.exp(
        total_negative_log_likelihood / total_predicted_tokens
    )

    result = {
        "model": str(model_path),
        "dataset": "Salesforce/wikitext:wikitext-2-raw-v1",
        "sequence_length": sequence_length,
        "evaluated_segments": segment_count,
        "evaluated_predicted_tokens": total_predicted_tokens,
        "perplexity": perplexity,
        "full_test_set": max_segments is None,
    }

    release_model(model)
    del tokenizer, encoded, test_data
    gc.collect()

    return result


# Dense WikiText-2 perplexity

This evaluates the unpruned HGRN-340M checkpoint before SparseGPT is run.

The quick-test limit is controlled by `PPL_MAX_SEGMENTS`. Set it to `None` for the complete test set.


In [14]:
RUN_DENSE_WIKITEXT_PPL = True

dense_ppl_result = None

if RUN_DENSE_WIKITEXT_PPL:
    dense_ppl_result = evaluate_wikitext2_perplexity(
        MODEL_ID,
        sequence_length=PPL_SEQUENCE_LENGTH,
        max_segments=PPL_MAX_SEGMENTS,
    )

    dense_ppl_path = (
        EVALUATION_DIR / "dense_wikitext2_perplexity.json"
    )
    dense_ppl_path.write_text(
        json.dumps(dense_ppl_result, indent=2),
        encoding="utf-8",
    )

    print()
    print(
        "Dense WikiText-2 perplexity:",
        f"{dense_ppl_result['perplexity']:.4f}",
    )
    print("Saved:", dense_ppl_path)
else:
    print("Dense WikiText-2 perplexity skipped.")


config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  WikiText-2 segments: 4/16
  WikiText-2 segments: 8/16
  WikiText-2 segments: 12/16
  WikiText-2 segments: 16/16

Dense WikiText-2 perplexity: 29.8597
Saved: /content/sparsegpt/sparse_hgrn/evaluation_results/dense_wikitext2_perplexity.json


## Run the HGRN-340M quick test

This runs all 24 HGRN blocks at 50% unstructured sparsity, but uses only eight WikiText-2 calibration sequences.

That is enough to check whether the complete SparseGPT pipeline works. It is not intended as a publication-quality calibration run.

For a stronger experiment after this succeeds, change:

```python
NSAMPLES = 32
```

or:

```python
NSAMPLES = 128
```


In [15]:
import subprocess
import sys

pruning_command = [
    sys.executable,
    str(REPO_DIR / "hgrn.py"),
    MODEL_ID,
    CALIBRATION_DATASET,
    "--sparsity",
    str(SPARSITY),
    "--nsamples",
    str(NSAMPLES),
    "--seed",
    str(SEED),
    "--percdamp",
    str(PERCDAMP),
    "--blocksize",
    str(BLOCKSIZE),
    "--save",
    str(SPARSE_MODEL_DIR),
    *EXTRA_FLAGS,
]

print("Running:")
print(" ".join(pruning_command))

subprocess.run(
    pruning_command,
    cwd=REPO_DIR,
    check=True,
)


Running:
/usr/bin/python3 /content/sparsegpt/hgrn.py m-a-p/340M-20B-HGRN-pure-baseline wikitext2 --sparsity 0.5 --nsamples 8 --seed 0 --percdamp 0.01 --blocksize 128 --save /content/sparsegpt/sparse_hgrn/hgrn-340m-sparsegpt-50


CompletedProcess(args=['/usr/bin/python3', '/content/sparsegpt/hgrn.py', 'm-a-p/340M-20B-HGRN-pure-baseline', 'wikitext2', '--sparsity', '0.5', '--nsamples', '8', '--seed', '0', '--percdamp', '0.01', '--blocksize', '128', '--save', '/content/sparsegpt/sparse_hgrn/hgrn-340m-sparsegpt-50'], returncode=0)

## Save the tokenizer with the sparse model

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)
tokenizer.save_pretrained(SPARSE_MODEL_DIR)

print("Tokenizer saved to:", SPARSE_MODEL_DIR)


Tokenizer saved to: /content/sparsegpt/sparse_hgrn/hgrn-340m-sparsegpt-50


# Sparse WikiText-2 perplexity

This uses the same evaluator and settings as the dense checkpoint, so the dense-versus-sparse comparison is controlled.

The `hgrn.py` pruning cell above also prints its built-in WikiText-2, PTB, and C4 perplexities. This section provides a separate WikiText-2 result saved in machine-readable form.


In [17]:
RUN_SPARSE_WIKITEXT_PPL = True

sparse_ppl_result = None

if RUN_SPARSE_WIKITEXT_PPL:
    if not SPARSE_MODEL_DIR.exists():
        raise FileNotFoundError(
            "The sparse checkpoint does not exist. "
            "Run the pruning cell first."
        )

    sparse_ppl_result = evaluate_wikitext2_perplexity(
        SPARSE_MODEL_DIR,
        sequence_length=PPL_SEQUENCE_LENGTH,
        max_segments=PPL_MAX_SEGMENTS,
    )

    sparse_ppl_path = (
        EVALUATION_DIR / "sparse_wikitext2_perplexity.json"
    )
    sparse_ppl_path.write_text(
        json.dumps(sparse_ppl_result, indent=2),
        encoding="utf-8",
    )

    print()
    print(
        "Sparse WikiText-2 perplexity:",
        f"{sparse_ppl_result['perplexity']:.4f}",
    )
    print("Saved:", sparse_ppl_path)
else:
    print("Sparse WikiText-2 perplexity skipped.")


  WikiText-2 segments: 4/16
  WikiText-2 segments: 8/16
  WikiText-2 segments: 12/16
  WikiText-2 segments: 16/16

Sparse WikiText-2 perplexity: 34.2298
Saved: /content/sparsegpt/sparse_hgrn/evaluation_results/sparse_wikitext2_perplexity.json


## Dense versus sparse perplexity

In [18]:
import pandas as pd

if dense_ppl_result and sparse_ppl_result:
    perplexity_comparison = pd.DataFrame(
        [
            {
                "checkpoint": "Dense",
                "perplexity": dense_ppl_result["perplexity"],
                "segments": dense_ppl_result["evaluated_segments"],
                "sequence_length": (
                    dense_ppl_result["sequence_length"]
                ),
            },
            {
                "checkpoint": "SparseGPT 50%",
                "perplexity": sparse_ppl_result["perplexity"],
                "segments": sparse_ppl_result["evaluated_segments"],
                "sequence_length": (
                    sparse_ppl_result["sequence_length"]
                ),
            },
        ]
    )

    perplexity_comparison["ppl_increase_vs_dense"] = (
        perplexity_comparison["perplexity"]
        - perplexity_comparison.loc[
            perplexity_comparison["checkpoint"] == "Dense",
            "perplexity",
        ].iloc[0]
    )

    display(perplexity_comparison)
else:
    print(
        "Run both perplexity cells to create the comparison table."
    )


,checkpoint,perplexity,segments,sequence_length,ppl_increase_vs_dense
0,Dense,29.859674,16,512,0.000000
1,SparseGPT 50%,34.229787,16,512,4.370112


# LM Evaluation Harness

This section evaluates zero-shot downstream tasks:

- `lambada_openai`
- `piqa`
- `arc_easy`
- `arc_challenge`

The HGRN model type is registered with Transformers, loaded through FLA, and passed to LM Evaluation Harness as a pre-initialized Hugging Face causal language model.

For the quick test, `LM_EVAL_LIMIT = 100` uses at most 100 examples per task. Set it to `None` for full results.

The full evaluations can take considerably longer than the limited test.


In [19]:
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import handle_non_serializable


def run_hgrn_lm_eval(
    model_path,
    output_filename,
    tasks=LM_EVAL_TASKS,
    limit=LM_EVAL_LIMIT,
):
    model, tokenizer = load_hgrn_checkpoint(model_path)

    evaluation_model = HFLM(
        pretrained=model,
        tokenizer=tokenizer,
        batch_size=LM_EVAL_BATCH_SIZE,
        max_length=LM_EVAL_MAX_LENGTH,
    )

    results = lm_eval.simple_evaluate(
        model=evaluation_model,
        tasks=tasks,
        num_fewshot=LM_EVAL_NUM_FEWSHOT,
        batch_size=LM_EVAL_BATCH_SIZE,
        limit=limit,
        log_samples=False,
    )

    output_path = EVALUATION_DIR / output_filename
    with output_path.open("w", encoding="utf-8") as output_file:
        json.dump(
            results,
            output_file,
            default=handle_non_serializable,
            indent=2,
        )

    del evaluation_model
    release_model(model)
    del tokenizer
    gc.collect()

    return results, output_path


def extract_lm_eval_metrics(results, checkpoint_name):
    rows = []

    for task_name, metrics in results["results"].items():
        for metric_name, value in metrics.items():
            if metric_name.endswith(",stderr"):
                continue
            if not isinstance(value, (int, float)):
                continue

            rows.append(
                {
                    "checkpoint": checkpoint_name,
                    "task": task_name,
                    "metric": metric_name,
                    "value": value,
                }
            )

    return rows


## Dense zero-shot LM-eval results

In [20]:
RUN_DENSE_LM_EVAL = True

dense_lm_eval_results = None
dense_lm_eval_path = None

if RUN_DENSE_LM_EVAL:
    dense_lm_eval_results, dense_lm_eval_path = (
        run_hgrn_lm_eval(
            MODEL_ID,
            "dense_lm_eval_results.json",
        )
    )

    dense_rows = extract_lm_eval_metrics(
        dense_lm_eval_results,
        "Dense",
    )
    display(pd.DataFrame(dense_rows))
    print("Saved:", dense_lm_eval_path)
else:
    print("Dense LM Evaluation Harness run skipped.")


README.md: 0.00B [00:00, ?B/s]

default/test/default.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

piqa_train.parquet:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

piqa_validation.parquet:   0%|          | 0.00/300k [00:00<?, ?B/s]

...

Running loglikelihood requests: 100%|██████████| 1099/1099 [00:18<00:00, 58.68it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:00<00:00, 107.77it/s]


,checkpoint,task,metric,value
0,Dense,lambada_openai,sample_len,100.000000
1,Dense,lambada_openai,"perplexity,none",50.239264
2,Dense,lambada_openai,"perplexity_stderr,none",13.621631
3,Dense,lambada_openai,"acc,none",0.240000
4,Dense,lambada_openai,"acc_stderr,none",0.042923
5,Dense,piqa,sample_len,100.000000
6,Dense,piqa,"acc,none",0.730000
7,Dense,piqa,"acc_stderr,none",0.044620
8,Dense,piqa,"acc_norm,none",0.720000
9,Dense,piqa,"acc_norm_stderr,none",0.045126


Saved: /content/sparsegpt/sparse_hgrn/evaluation_results/dense_lm_eval_results.json


## Sparse zero-shot LM-eval results

In [21]:
RUN_SPARSE_LM_EVAL = True

sparse_lm_eval_results = None
sparse_lm_eval_path = None

if RUN_SPARSE_LM_EVAL:
    if not SPARSE_MODEL_DIR.exists():
        raise FileNotFoundError(
            "The sparse checkpoint does not exist. "
            "Run the pruning cell first."
        )

    sparse_lm_eval_results, sparse_lm_eval_path = (
        run_hgrn_lm_eval(
            SPARSE_MODEL_DIR,
            "sparse_lm_eval_results.json",
        )
    )

    sparse_rows = extract_lm_eval_metrics(
        sparse_lm_eval_results,
        "SparseGPT 50%",
    )
    display(pd.DataFrame(sparse_rows))
    print("Saved:", sparse_lm_eval_path)
else:
    print("Sparse LM Evaluation Harness run skipped.")


Running loglikelihood requests: 100%|██████████| 1099/1099 [00:23<00:00, 46.57it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:00<00:00, 106.26it/s]


,checkpoint,task,metric,value
0,SparseGPT 50%,lambada_openai,sample_len,100.000000
1,SparseGPT 50%,lambada_openai,"perplexity,none",97.892231
2,SparseGPT 50%,lambada_openai,"perplexity_stderr,none",26.863093
3,SparseGPT 50%,lambada_openai,"acc,none",0.160000
4,SparseGPT 50%,lambada_openai,"acc_stderr,none",0.036845
5,SparseGPT 50%,piqa,sample_len,100.000000
6,SparseGPT 50%,piqa,"acc,none",0.680000
7,SparseGPT 50%,piqa,"acc_stderr,none",0.046883
8,SparseGPT 50%,piqa,"acc_norm,none",0.690000
9,SparseGPT 50%,piqa,"acc_norm_stderr,none",0.046482


Saved: /content/sparsegpt/sparse_hgrn/evaluation_results/sparse_lm_eval_results.json


## Dense versus sparse LM-eval comparison

In [22]:
if dense_lm_eval_results and sparse_lm_eval_results:
    dense_rows = extract_lm_eval_metrics(
        dense_lm_eval_results,
        "Dense",
    )
    sparse_rows = extract_lm_eval_metrics(
        sparse_lm_eval_results,
        "SparseGPT 50%",
    )

    lm_eval_comparison = pd.DataFrame(
        dense_rows + sparse_rows
    )

    comparison_table = lm_eval_comparison.pivot_table(
        index=["task", "metric"],
        columns="checkpoint",
        values="value",
    ).reset_index()

    if (
        "Dense" in comparison_table.columns
        and "SparseGPT 50%" in comparison_table.columns
    ):
        comparison_table["sparse_minus_dense"] = (
            comparison_table["SparseGPT 50%"]
            - comparison_table["Dense"]
        )

    display(comparison_table)
else:
    print(
        "Run both dense and sparse LM-eval cells "
        "to create the comparison table."
    )


checkpoint,task,metric,Dense,SparseGPT 50%,sparse_minus_dense
0,arc_challenge,"acc,none",0.250000,0.220000,-0.030000
1,arc_challenge,"acc_norm,none",0.250000,0.250000,0.000000
2,arc_challenge,"acc_norm_stderr,none",0.043519,0.043519,0.000000
3,arc_challenge,"acc_stderr,none",0.043519,0.041633,-0.001886
4,arc_challenge,sample_len,100.000000,100.000000,0.000000
5,arc_easy,"acc,none",0.580000,0.490000,-0.090000
6,arc_easy,"acc_norm,none",0.570000,0.490000,-0.080000
7,arc_easy,"acc_norm_stderr,none",0.049757,0.050242,0.000485
8,arc_easy,"acc_stderr,none",0.049604,0.050242,0.000637
9,arc_easy,sample_len,100.000000,100.000000,0.000000


## Inspect the saved checkpoint

In [23]:
if not SPARSE_MODEL_DIR.exists():
    raise FileNotFoundError(
        "The sparse model directory does not exist. Run pruning first."
    )

for path in sorted(SPARSE_MODEL_DIR.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name:45s} {size_mb:10.2f} MiB")


config.json                                         0.00 MiB
generation_config.json                              0.00 MiB
model.safetensors                                1301.40 MiB
special_tokens_map.json                             0.00 MiB
tokenizer.json                                      3.34 MiB
tokenizer_config.json                               0.00 MiB


## Verify the saved zero-weight sparsity

This reloads the sparse checkpoint on CPU and counts zeros in the HGRN block linear layers.

Embeddings, normalization parameters, and the language-model head are excluded because the repository-style wrapper prunes the recurrent blocks.


In [24]:
import gc
import torch
import torch.nn as nn
import fla

from fla.models.hgrn import HGRNConfig, HGRNForCausalLM
from transformers import AutoConfig, AutoModelForCausalLM

AutoConfig.register(
    "hgrn",
    HGRNConfig,
    exist_ok=True,
)
AutoModelForCausalLM.register(
    HGRNConfig,
    HGRNForCausalLM,
    exist_ok=True,
)

saved_model = AutoModelForCausalLM.from_pretrained(
    SPARSE_MODEL_DIR,
    torch_dtype="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

zero_count = 0
weight_count = 0
per_module = []

for name, module in saved_model.named_modules():
    if isinstance(module, nn.Linear) and name.startswith("model.layers."):
        zeros = int((module.weight == 0).sum().item())
        total = module.weight.numel()

        zero_count += zeros
        weight_count += total
        per_module.append((name, zeros / total))

print(f"Selected linear-layer sparsity: {zero_count / weight_count:.4%}")
print(f"Zero weights: {zero_count:,}/{weight_count:,}")
print()
print("First ten modules:")

for name, sparsity in per_module[:10]:
    print(f"{name:55s} {sparsity:.4%}")

del saved_model
gc.collect()
torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!


Selected linear-layer sparsity: 50.0005%
Zero weights: 154,142,355/308,281,344

First ten modules:
model.layers.0.attn.i_proj                              50.0008%
model.layers.0.attn.f_proj                              50.0008%
model.layers.0.attn.g_proj                              50.0008%
model.layers.0.attn.o_proj                              50.0008%
model.layers.0.mlp.gate_proj                            50.0003%
model.layers.0.mlp.up_proj                              50.0003%
model.layers.0.mlp.down_proj                            50.0008%
model.layers.1.attn.i_proj                              50.0008%
model.layers.1.attn.f_proj                              50.0008%
model.layers.1.attn.g_proj                              50.0008%


# Compare generations

The following cells generate text from the dense and sparse models.

Only one model is placed on the GPU at a time to reduce memory usage.


In [25]:
import gc
import torch
import fla

from fla.models.hgrn import HGRNConfig, HGRNForCausalLM
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
)

AutoConfig.register(
    "hgrn",
    HGRNConfig,
    exist_ok=True,
)
AutoModelForCausalLM.register(
    HGRNConfig,
    HGRNForCausalLM,
    exist_ok=True,
)

DEVICE = "cuda"
DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

generation_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
if generation_tokenizer.pad_token_id is None:
    generation_tokenizer.pad_token_id = (
        generation_tokenizer.eos_token_id
    )

print("Generation dtype:", DTYPE)


Generation dtype: torch.bfloat16


In [26]:
INPUT_TEXT = "It takes a great deal of bravery"
MAX_NEW_TOKENS = 40

inputs = generation_tokenizer(
    INPUT_TEXT,
    return_tensors="pt",
).to(DEVICE)


## Completion by the dense model

In [27]:
dense_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
dense_model.eval()

with torch.inference_mode():
    dense_output = dense_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=generation_tokenizer.pad_token_id,
    )

print(
    generation_tokenizer.decode(
        dense_output[0].cpu(),
        skip_special_tokens=True,
    )
)

del dense_model, dense_output
gc.collect()
torch.cuda.empty_cache()


It takes a great deal of bravery to be a fighter pilot. The first time you fly a plane, you are in a very different world than you are in the first time you fly a plane. You are in a different environment,


## Completion by the sparse model

In [28]:
sparse_model = AutoModelForCausalLM.from_pretrained(
    SPARSE_MODEL_DIR,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
sparse_model.eval()

with torch.inference_mode():
    sparse_output = sparse_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=generation_tokenizer.pad_token_id,
    )

print(
    generation_tokenizer.decode(
        sparse_output[0].cpu(),
        skip_special_tokens=True,
    )
)

del sparse_model, sparse_output
gc.collect()
torch.cuda.empty_cache()


It takes a great deal of bravery to be a fighter pilot. The fighter pilot is the most important fighter pilot in the world. He is the pilot who is responsible for the entire mission. He is the one who is responsible for the


# Additional command examples

### 2:4 SparseGPT

```bash
python hgrn.py m-a-p/340M-20B-HGRN-pure-baseline c4 \
    --prunen 2 --prunem 4 --nsamples 8 \
    --save sparse_hgrn/hgrn-340m-2of4
```

### Magnitude pruning baseline

```bash
python hgrn.py m-a-p/340M-20B-HGRN-pure-baseline c4 \
    --sparsity 0.5 --gmp
```

### Prune only the first HGRN block

```bash
python hgrn.py m-a-p/340M-20B-HGRN-pure-baseline wikitext2 \
    --sparsity 0.5 --nsamples 4 --maxlayer 1
```

### True-sequential SparseGPT

```bash
python hgrn.py m-a-p/340M-20B-HGRN-pure-baseline c4 \
    --sparsity 0.5 --nsamples 8 --true-sequential \
    --save sparse_hgrn/hgrn-340m-true-sequential
```

> The saved tensors keep their original dense shapes and contain zero values. The checkpoint therefore will not automatically become smaller or faster without a sparse storage format and sparse-aware inference kernels.
